# GraphRAG Pipeline - Qwen Only (Dual GPU T4 x2)

## Configuration
- **Model**: Qwen/Qwen3-4B (Local only, NO DeepSeek API)
- **GPU 0**: Main Qwen model
- **GPU 1**: Embedding + Reranker models

## Setup
1. Select **GPU T4 x2** in Kaggle Settings
2. Add `NEO4J_PASSWORD` to Kaggle Secrets
3. Run all cells

In [ ]:
# Install dependencies
!pip install -q transformers torch accelerate neo4j bitsandbytes sentence-transformers requests google-generativeai

In [ ]:
import json
import logging
import os
import re
import time
import gc
import torch
import numpy as np
from typing import List, Dict, Any, Tuple, Optional
from tqdm import tqdm
from datetime import datetime

# ================================================================================
# MULTI-GPU SETUP
# ================================================================================
print("="*70)
print("GPU CONFIGURATION")
print("="*70)

num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"Number of GPUs: {num_gpus}")

for i in range(num_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name}")
    print(f"          Memory: {props.total_memory / 1024**3:.1f} GB")
    print(f"          Free: {(props.total_memory - torch.cuda.memory_allocated(i)) / 1024**3:.1f} GB")

if num_gpus >= 2:
    print("\n>>> MULTI-GPU MODE ENABLED <<<")
    print("  GPU 0: Qwen Model (Main)")
    print("  GPU 1: Embedding + Reranker")
    for i in range(num_gpus):
        with torch.cuda.device(i):
            torch.cuda.empty_cache()
elif num_gpus == 1:
    print("\nSingle GPU mode")
    torch.cuda.empty_cache()
else:
    print("\nWARNING: No GPU available!")

# Enable optimizations
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('medium')

print("="*70)

# Kaggle secrets
try:
    from kaggle_secrets import UserSecretsClient
    _KAGGLE_SECRETS_AVAILABLE = True
except:
    _KAGGLE_SECRETS_AVAILABLE = False

def get_secret(key: str, default: str = None) -> str:
    if _KAGGLE_SECRETS_AVAILABLE:
        try:
            user_secrets = UserSecretsClient()
            val = user_secrets.get_secret(key)
            if val is not None and val != "":
                return val
        except:
            pass
    val = os.getenv(key)
    return val if val else default

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("GraphRAG")

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

In [ ]:
# ================================================================================
# IMPORT GRAPHRAG PACKAGE
# ================================================================================

import sys
sys.path.insert(0, '/kaggle/input/historical')

from graphrag.core import GraphRAGConfig
from graphrag.neo4j_manager import Neo4jManager
from graphrag.embeddings import EmbeddingGenerator, Reranker
from graphrag.retriever import HybridRetriever, ContextBuilder
from graphrag.pipeline import (
    EntityExtractor,
    AnswerGenerator,
    GraphRAGPipeline
)

print("GraphRAG package imported!")

In [ ]:
# ================================================================================
# CONFIGURATION - QWEN ONLY (NO DEEPSEEK)
# ================================================================================

QUESTION_FILE = "/kaggle/input/historical/sample_20question.json"
KG_FILE = "/kaggle/input/historical/knowledge_graph_historical_v5.json"
ENTITIES_FILE = "/kaggle/input/historical/entities_v5.json"
OUTPUT_FILE = "/kaggle/working/results_qwen_only.json"

NEO4J_PASSWORD = get_secret("NEO4J_PASSWORD", "password")

# Create config - QWEN ONLY, NO DEEPSEEK
config = GraphRAGConfig(
    neo4j_uri=os.getenv("NEO4J_URI", "neo4j+s://5f398723.databases.neo4j.io"),
    neo4j_user="neo4j",
    neo4j_password=NEO4J_PASSWORD,
    allow_neo4j_fallback=True,
    
    # DISABLE DEEPSEEK - USE QWEN ONLY
    use_deepseek_api=False,
    deepseek_api_key="",  # Empty = no DeepSeek
    
    # Qwen model settings
    qwen_model="Qwen/Qwen3-4B",
    
    # Files
    questions_file=QUESTION_FILE,
    kg_file=KG_FILE,
    entities_file=ENTITIES_FILE,
    output_file=OUTPUT_FILE,
)

print("="*70)
print("CONFIGURATION - QWEN ONLY MODE")
print("="*70)
print(f"Model:     {config.qwen_model}")
print(f"DeepSeek:  DISABLED")
print(f"Neo4j:     {config.neo4j_uri}")
print(f"GPUs:      {num_gpus}")
print("="*70)

In [ ]:
# ================================================================================
# INITIALIZE PIPELINE - QWEN ONLY with MULTI-GPU
# ================================================================================

print("\n" + "="*70)
print("INITIALIZING PIPELINE (Qwen Only + Multi-GPU)")
print("="*70)

# Clear GPU cache before loading
for i in range(num_gpus):
    with torch.cuda.device(i):
        torch.cuda.empty_cache()
gc.collect()

# Print GPU memory before
for i in range(num_gpus):
    alloc = torch.cuda.memory_allocated(i) / 1024**3
    print(f"GPU {i} before load: {alloc:.2f} GB")

# Initialize pipeline - NO DUAL MODE (Qwen only)
pipeline = GraphRAGPipeline(
    config=config,
    use_dual_mode=False  # QWEN ONLY - no dual mode
)

# Print GPU memory after
for i in range(num_gpus):
    alloc = torch.cuda.memory_allocated(i) / 1024**3
    print(f"GPU {i} after load: {alloc:.2f} GB")

print("\nPipeline Status:")
print(f"  Neo4j Connected:  {pipeline.neo4j_connected}")
print(f"  Graph Encoder:    {'Enabled' if getattr(pipeline, 'graph_encoder', None) else 'Disabled'}")
print(f"  Embedding Fusion: {'Enabled' if getattr(pipeline, 'embedding_fusion', None) else 'Disabled'}")
print(f"  Trust Calculator: {'Enabled' if getattr(pipeline, 'trust_calculator', None) else 'Disabled'}")
print(f"  Model:            Qwen (Local Only)")
print("="*70)

In [ ]:
# ================================================================================
# LOAD QUESTIONS
# ================================================================================

with open(QUESTION_FILE, 'r', encoding='utf-8') as f:
    data = json.load(f)

questions = data if isinstance(data, list) else data.get("multiple_choice", []) + data.get("true_false", [])

print(f"Loaded {len(questions)} questions")
print(f"  MCQ: {sum(1 for q in questions if len(q.get('options', [])) > 2)}")
print(f"  T/F: {sum(1 for q in questions if len(q.get('options', [])) == 2)}")

# Uncomment to limit for testing:
# questions = questions[:20]

In [ ]:
# ================================================================================
# PROCESS QUESTIONS - QWEN ONLY with GPU MONITORING
# ================================================================================

results = []
stats = {
    "total": 0, "correct": 0,
    "mcq": 0, "mcq_correct": 0,
    "tf": 0, "tf_correct": 0
}

start_time = time.time()

print("\n" + "="*70)
print("PROCESSING QUESTIONS - QWEN ONLY")
print("="*70)

for i, q in enumerate(tqdm(questions, desc="Processing")):
    try:
        # Clear cache periodically on all GPUs
        if i % 20 == 0:
            for g in range(num_gpus):
                with torch.cuda.device(g):
                    torch.cuda.empty_cache()
            gc.collect()
        
        # Process question
        result = pipeline.process_question(q)
        
        # Format output (Qwen only - no dual results)
        formatted = {
            "question": result.get("question", ""),
            "question_type": result.get("question_type", ""),
            "correct_answer": result.get("correct_answer", ""),
            "entities_extracted": result.get("entities_extracted", []),
            "context_length": result.get("context_length", 0),
            "model_answer": result.get("model_answer", ""),
            "raw_response": result.get("raw_response", ""),
            "is_correct": result.get("is_correct", False),
            "evidence": result.get("evidence", []),
            "processing_time": result.get("processing_time_seconds", 0)
        }
        results.append(formatted)
        
        # Update stats
        stats["total"] += 1
        q_type = formatted["question_type"]
        if q_type in ["mcq", "tf"]:
            stats[q_type] += 1
            if formatted["is_correct"]:
                stats["correct"] += 1
                stats[f"{q_type}_correct"] += 1
        
        # Progress update
        if (i + 1) % 20 == 0 or i == len(questions) - 1:
            acc = stats["correct"] / stats["total"] * 100 if stats["total"] > 0 else 0
            gpu_info = " | ".join([f"GPU{g}:{torch.cuda.memory_allocated(g)/1024**3:.1f}GB" for g in range(num_gpus)])
            print(f"\n[{i+1}/{len(questions)}] Accuracy: {acc:.1f}% | {gpu_info}")
            
    except Exception as e:
        print(f"Error Q{i+1}: {e}")
        import traceback
        traceback.print_exc()

total_time = time.time() - start_time
print(f"\nCompleted in {total_time:.1f}s ({total_time/len(questions):.2f}s/question)")

In [ ]:
# ================================================================================
# FINAL RESULTS - QWEN ONLY
# ================================================================================

print("\n" + "="*70)
print("FINAL RESULTS - QWEN ONLY")
print("="*70)

total = stats["total"]
acc = stats["correct"] / total * 100 if total > 0 else 0

print(f"\nOVERALL: {stats['correct']}/{total} = {acc:.2f}%")

if stats["mcq"] > 0:
    mcq_acc = stats["mcq_correct"] / stats["mcq"] * 100
    print(f"MCQ:     {stats['mcq_correct']}/{stats['mcq']} = {mcq_acc:.2f}%")

if stats["tf"] > 0:
    tf_acc = stats["tf_correct"] / stats["tf"] * 100
    print(f"T/F:     {stats['tf_correct']}/{stats['tf']} = {tf_acc:.2f}%")

print(f"\nProcessing Time: {total_time:.1f}s ({total_time/len(questions):.2f}s/question)")
print(f"GPUs Used: {num_gpus}")
print("="*70)

In [ ]:
# ================================================================================
# SAVE RESULTS
# ================================================================================

mcq_acc = (stats["mcq_correct"] / stats["mcq"] * 100) if stats["mcq"] > 0 else 0
tf_acc = (stats["tf_correct"] / stats["tf"] * 100) if stats["tf"] > 0 else 0

output = {
    "generated_at": datetime.now().isoformat(),
    "pipeline": "GraphRAG Qwen Only (Multi-GPU)",
    "config": {
        "model": config.qwen_model,
        "num_gpus": num_gpus,
        "neo4j_connected": pipeline.neo4j_connected,
    },
    "stats": {
        "total": stats["total"],
        "mcq": stats["mcq"],
        "tf": stats["tf"],
    },
    "accuracy": {
        "total": round(acc, 2),
        "mcq": round(mcq_acc, 2),
        "tf": round(tf_acc, 2),
    },
    "processing_time_seconds": round(total_time, 2),
    "results": results
}

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Results saved to: {OUTPUT_FILE}")

In [ ]:
# ================================================================================
# CLEANUP
# ================================================================================

# Close Neo4j connection
if hasattr(pipeline, 'neo4j') and pipeline.neo4j:
    try:
        pipeline.close()
        print("Neo4j connection closed.")
    except:
        pass

# Final GPU cleanup
for i in range(num_gpus):
    with torch.cuda.device(i):
        torch.cuda.empty_cache()
gc.collect()

# Print final GPU memory
for i in range(num_gpus):
    alloc = torch.cuda.memory_allocated(i) / 1024**3
    print(f"GPU {i} final: {alloc:.2f} GB")

print("\n" + "="*70)
print("COMPLETED!")
print("="*70)
print(f"Model:    {config.qwen_model}")
print(f"Accuracy: {acc:.2f}%")
print(f"Time:     {total_time:.1f}s")
print(f"Output:   {OUTPUT_FILE}")